# Graphs

*Represent a collaboration network, distinguish direct and indirect connections, and compare adjacency lists with matrices.*

A **graph** represents items as nodes and connections as edges. In a student collaboration network, the student IDs identify the nodes, while the edges record who has worked with whom. A flat set of IDs would lose those connections.

A **cycle** follows edges back to the starting node without repeating other nodes. A tree is a connected undirected graph with no cycles; general graphs can contain cycles. We define one collaboration graph in this notebook and reuse it to inspect neighbors, find indirect connections, and build an adjacency matrix.


## Nodes, Edges, and Degree

In the collaboration graph, a node is a student and an edge means two students have worked together. A graph can have cycles, several paths between nodes, or disconnected groups. It does not require a root or a single parent for each item.

An **undirected** edge connects both endpoints equally. A **directed** edge instead has a source and destination, as in a link from a citing paper to a cited paper. A **weight** can record a quantity such as collaboration count. Our prepared graph is undirected and unweighted, with no self-edges or repeated neighbors.

An **adjacency list** maps each node to its neighbors. Here each edge appears at both ends: listing S02 under S01 requires listing S01 under S02. A node's **degree** is its number of neighbors, and dividing the total neighbor count by two gives the number of undirected edges.


**Think before running or viewing the saved output.** S01 connects to S02 and S03, S02 connects to S03, and S03 connects to S04. Is this a tree? What is S03's degree?


<img src="https://raw.githubusercontent.com/sonamu-jun/introduction-to-bigdata/main/02-2_Data_Structures/assets/08_graphs/collaboration_graph.webp" width="340" alt="Four nodes form edges S01-S02, S01-S03, S02-S03, and S03-S04. The first three edges form a cycle. S03 has three neighbors.">

Amber marks the S01-S02-S03 cycle. S03 also connects to S04.


In [1]:
neighbors_by_student = {
    "S01": ["S02", "S03"],
    "S02": ["S01", "S03"],
    "S03": ["S01", "S02", "S04"],
    "S04": ["S03"],
}

neighbor_entry_count = sum(len(neighbors) for neighbors in neighbors_by_student.values())
graph_edge_count = neighbor_entry_count // 2

print("Neighbors of S03:", neighbors_by_student["S03"])
print("Node count:", len(neighbors_by_student))
print("Neighbor entries:", neighbor_entry_count)
print("Undirected edge count:", graph_edge_count)
for student_id in sorted(neighbors_by_student):
    print(f"{student_id}: degree={len(neighbors_by_student[student_id])}")


Neighbors of S03: ['S01', 'S02', 'S04']
Node count: 4
Neighbor entries: 8
Undirected edge count: 4
S01: degree=2
S02: degree=2
S03: degree=3
S04: degree=1


**Check:** S03 has degree three. The cycle `S01 -> S02 -> S03 -> S01` means the graph is not a tree. There are four nodes and four edges, represented by eight neighbor entries.

S01 and S02 each have degree two; S04 has degree one. S03 has the most direct collaborators in this small network. Because the edges are unweighted, degree does not tell us how often those students worked together.


## Direct and Indirect Connections

A direct connection has one edge. An indirect connection follows a path through other nodes. To find common collaborators, convert two neighbor lists to sets and take their intersection with `&`. Each common neighbor supplies a two-edge path between the selected students.

We reuse S01 and S04 from the same graph. The membership check asks whether they have a direct edge; the intersection checks whether they share an intermediary. Sort the set before printing for a fixed output order.


In [2]:
first_neighbors = set(neighbors_by_student["S01"])
second_neighbors = set(neighbors_by_student["S04"])
shared_neighbors = first_neighbors & second_neighbors

print("S01 neighbors:", sorted(first_neighbors))
print("S04 neighbors:", sorted(second_neighbors))
print("Shared collaborators:", sorted(shared_neighbors))
print("Direct S01-S04 connection:", "S04" in neighbors_by_student["S01"])


S01 neighbors: ['S02', 'S03']
S04 neighbors: ['S03']
Shared collaborators: ['S03']
Direct S01-S04 connection: False


S01 and S04 have no direct edge, but both connect to S03. This gives the two-edge path `S01 -> S03 -> S04`. There is also a longer path through S02: `S01 -> S02 -> S03 -> S04`.

A missing direct edge therefore does not mean two students are disconnected. Shared-neighbor intersection detects two-edge paths; it does not search for every possible longer path.


## Adjacency Matrices

The same graph can be represented as a square matrix. Fix the row and column node order first. Entry `[i][j]` is `1` when the corresponding students have a direct edge and `0` otherwise. `int()` converts the membership result from `True` or `False` to `1` or `0`.

The outer comprehension creates a row for each source node; the inner comprehension checks each possible target. `zip()` pairs each node label with its completed row for printing. The matrix and the neighbor lists preserve the same links.


In [3]:
node_order = sorted(neighbors_by_student)
adjacency_matrix = [
    [int(target in neighbors_by_student[source]) for target in node_order]
    for source in node_order
]

print("Row and column order:", node_order)
for student_id, row in zip(node_order, adjacency_matrix):
    print(student_id, row)
print("Direct S01-S04 connection:", adjacency_matrix[0][3] == 1)
print("Direct S03-S04 connection:", adjacency_matrix[2][3] == 1)


Row and column order: ['S01', 'S02', 'S03', 'S04']
S01 [0, 1, 1, 0]
S02 [1, 0, 1, 0]
S03 [1, 1, 0, 1]
S04 [0, 0, 1, 0]
Direct S01-S04 connection: False
Direct S03-S04 connection: True


S01's row is `[0, 1, 1, 0]`: it has direct edges to S02 and S03. The zero for S04 agrees with the previous result, even though an indirect path exists. The diagonal is zero because no student is connected to themself. The matrix is symmetric because every edge is undirected.

Keep `node_order` with the matrix: positions alone do not identify students. This matrix stores 16 entries, while the adjacency lists store eight neighbor entries. A matrix reserves `n × n` entries even when few edges exist; adjacency lists can avoid storing all those absent connections.


Use a graph to preserve general connections between items. Neighbor lists expose direct and shared connections; an adjacency matrix records the same edges under an explicit node order. A missing direct edge does not rule out an indirect path, and a flat set of node names cannot preserve either kind of connection.
